In [0]:
# Master Pipeline Orchestrator for Consumer Financial Complaint Analytics
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F

dbutils.widgets.dropdown("mode", "auto", ["auto", "manual"], "API Fetch Mode")
dbutils.widgets.text("start_date", "", "Manual Start Date (YYYY-MM-DD)")
dbutils.widgets.text("end_date", "", "Manual End Date (YYYY-MM-DD)")
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental", "backfill"], "Load Type")

RUN_MODE = dbutils.widgets.get("mode")
MANUAL_START = dbutils.widgets.get("start_date").strip()
MANUAL_END = dbutils.widgets.get("end_date").strip()
LOAD_TYPE = dbutils.widgets.get("load_type")

RUN_ID = str(uuid.uuid4())
START_TIME = datetime.now(timezone.utc)
LOGS_TABLE = "workspace.complaints.pipeline_execution_logs"

print("=" * 60)
print(f"PIPELINE RUN STARTED: {START_TIME.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"Mode: {RUN_MODE} | Load Type: {LOAD_TYPE} | Run ID: {RUN_ID}")
if RUN_MODE == "manual":
    print(f"Manual Window: {MANUAL_START} to {MANUAL_END}")
print("=" * 60)

status = "Success"
error_msg = None

try:
    # Step 1: Ingest live delta from CFPB API
    print("\n[Step 1/3] Ingesting complaints from CFPB API...")
    api_args = {"mode": RUN_MODE}
    if RUN_MODE == "manual":
        api_args["start_date"] = MANUAL_START
        api_args["end_date"] = MANUAL_END

    api_status = dbutils.notebook.run(
        "02_ingest_api",
        timeout_seconds=1800,
        arguments=api_args
    )
    print(f"API Ingestion Result: {api_status}")

    # Step 2: Append raw files to Bronze Delta Lake
    if api_status in ["UP_TO_DATE", "ALREADY_EXISTS"]:
        print(f"Skipping downstream steps: API reported {api_status}.")
    else:
        print("\n[Step 2/3] Ingesting Raw data into Bronze Delta table...")
        bronze_status = dbutils.notebook.run(
            "03_raw_to_bronze",
            timeout_seconds=1200,
            arguments={"load_type": LOAD_TYPE}
        )
        print(f"Bronze Ingestion Result: {bronze_status}")

        # Step 3: Clean, validate, quarantine, and merge into Silver Delta Lake
        print("\n[Step 3/3] Transforming Bronze to Silver (Idempotent MERGE)...")
        silver_status = dbutils.notebook.run(
            "04_bronze_to_silver",
            timeout_seconds=1800,
            arguments={"process_all": "true"}
        )
        print(f"Silver Transformation Result: {silver_status}")

except Exception as e:
    status = "Failure"
    error_msg = str(e)[:2000]
    print(f"\nPipeline run failed with error: {error_msg}")
    raise e

finally:
    END_TIME = datetime.now(timezone.utc)
    duration_seconds = round((END_TIME - START_TIME).total_seconds(), 2)

    if spark.catalog.tableExists(LOGS_TABLE):
        spark.createDataFrame([(
            RUN_ID, f"master_{START_TIME:%Y%m%d_%H%M%S}", "orchestrator", "master_run",
            f"mode={RUN_MODE}, load_type={LOAD_TYPE}",
            START_TIME, END_TIME, float(duration_seconds), status,
            0, 0, 0, 0, error_msg
        )], schema="""run_id string, batch_id string, layer string, load_type string, parameter string, 
                     start_time timestamp, end_time timestamp, duration_seconds double, status string, 
                     rows_read long, rows_inserted long, rows_updated long, rows_quarantined long, error_message string""") \
        .write.format("delta").mode("append").saveAsTable(LOGS_TABLE)

    print("\n" + "=" * 60)
    print(f"PIPELINE RUN FINISHED: {END_TIME.strftime('%Y-%m-%d %H:%M:%S UTC')}")
    print(f"Total Duration:        {duration_seconds} seconds ({round(duration_seconds / 60, 2)} minutes)")
    print(f"Final Status:          {status}")
    print("=" * 60)

In [0]:
# Display recent execution metrics and silver table record count
logs_table = "workspace.complaints.pipeline_execution_logs"
silver_table = "workspace.complaints.silver_complaints"
quarantine_table = "workspace.complaints.silver_quarantine"

print("--- Current Layer Counts ---")
print("Silver Total Records:     ", spark.table(silver_table).count())
print("Quarantine Total Records: ", spark.table(quarantine_table).count())

print("\n--- Latest Execution Logs ---")
display(
    spark.table(logs_table)
    .orderBy(F.col("start_time").desc())
    .limit(10)
)

In [0]:
# Preview normalized Silver analytical records
display(
    spark.table(silver_table)
    .select("complaint_id", "date_received", "company", "product", "issue", "state", "timely")
    .orderBy(F.col("date_received").desc())
    .limit(10)
)

In [0]:
display(spark.table("workspace.complaints.pipeline_execution_logs").orderBy(F.col("start_time").desc()).limit(5))

In [0]:
from pyspark.sql import functions as F

display(
    spark.table("workspace.complaints.silver_complaints")
    .groupBy(F.to_date("date_received").alias("date"))
    .count()
    .orderBy(F.col("date").desc())
)